# Configuration Tables Demo

This notebook shows how to load a Model Zoo configuration that sources
hyperparameters and coefficient priors from CSV tables. It reuses the marketing
mix example so you can see how table-driven edits propagate without touching the
JSON payload.

## Prerequisites

- The repository root must be available so we can reference `configs/`.
- `requests` is only needed if you want to POST the modified config to the API;
  the core of this notebook uses in-memory loaders.
- Optional: run `docker compose up --build` if you want to register the config
  with the FastAPI service and execute a run.

In [ ]:
#!pip install requests pandas --quiet

## Locate Config and Tables

`marketing_mix.json` references `marketing_mix_coefficients.csv` for its
coefficient priors. We load both so you can inspect the raw tables.

In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

CONFIG_PATH = (PROJECT_ROOT / "configs" / "marketing_mix.json").resolve()
COEFFS_PATH = (PROJECT_ROOT / "configs" / "marketing_mix_coefficients.csv").resolve()

print(f"Config path: {CONFIG_PATH}")
print(f"Coefficient table: {COEFFS_PATH}")

coeffs_df = pd.read_csv(COEFFS_PATH)
coeffs_df

## Load Configuration with Table Support

`load_model_configuration` automatically merges table rows when `tables` are
specified. The coefficient priors appear under `cfg.coefficients` and will
include both inline and table-sourced definitions.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / "configs" / "marketing_mix.json").resolve()
COEFFS_PATH = (PROJECT_ROOT / "configs" / "marketing_mix_coefficients.csv").resolve()

print(f"Config path: {CONFIG_PATH}")
print(f"Coefficient table: {COEFFS_PATH}")

coeffs_df = pd.read_csv(COEFFS_PATH)
coeffs_df

## Modify Tables Without Editing JSON

Here we build a temporary DataFrame that adds a new `digital` channel and pushes
it into the config via a table override. This leaves the original CSV untouched
by writing to a temp file.

In [ ]:
import pandas as pd

clean_table = coeffs_df.copy()
clean_table = clean_table[clean_table['name'] != 'radio']
clean_table = pd.concat([
    clean_table,
    pd.DataFrame([
        {
            'name': 'digital',
            'distribution': 'normal',
            'params.mu': 0.6,
            'params.sigma': 0.3,
            'constraints.sign': 'positive'
        }
    ])
], ignore_index=True)

def _row_to_jsonable(row: dict) -> dict:
    out = {}
    for key, value in row.items():
        if isinstance(value, float) and pd.isna(value):
            out[key] = None
        else:
            out[key] = value
    return out

table_rows = [_row_to_jsonable(row) for row in clean_table.to_dict(orient='records')]
print('Preview of modified table rows:')
table_rows[:3]


## Load a Customised Configuration

Clone the original config, point the `coefficient_priors.tables` entry to the
temporary CSV, and reload to see the new priors take effect.

In [ ]:
import json
from copy import deepcopy
from src.model_zoo.utils.config_loader import load_model_configuration_from_dict

config_data = json.loads(CONFIG_PATH.read_text())
config_mut = deepcopy(config_data)
priors_section = config_mut.setdefault('coefficient_priors', {})
priors_section['tables'] = [{
    'content': table_rows
}]

cfg_custom = load_model_configuration_from_dict(
    config_mut,
    env='dev',
    base_path=CONFIG_PATH.parent,
)

cfg_custom.coefficients.to_runner_params()


## Optional: Register and Run via API

If the API is running, register the modified config under a new environment and
submit a run. The run endpoint is asynchronous, so we poll until it completes.

### Adjust Hyperparameters

You can tune sampler settings before registering the configuration. The next
cell updates the number of draws and the sigma scale hyperparameter.

In [ ]:
values = config_mut.setdefault('hyperparameters', {}).setdefault('values', [])
model_id = config_mut['model']['id']

def _set_value(param: str, value, env: str | None = 'dev'):
    for entry in values:
        if entry.get('model_id') == model_id and entry.get('param') == param and entry.get('env') == env:
            entry['value'] = value
            return
    values.append({'model_id': model_id, 'param': param, 'value': value, 'env': env})

_set_value('draws', 600)
_set_value('sigma_scale', 3.0)
values

In [ ]:
import requests
import time

BASE_URL = "http://localhost:8000"
EXPERIMENT_ENV = "dev-table-demo"

register_payload = {
    'env': EXPERIMENT_ENV,
    'model_id': cfg_custom.model['id'],
    'config': config_mut,
}
reg_resp = requests.post(f"{BASE_URL.rstrip('/')}/registry/configs", json=register_payload, timeout=30)
reg_resp.raise_for_status()
print('Registered config for env', EXPERIMENT_ENV)

run_payload = {
    'model_id': cfg_custom.model['id'],
    'env': EXPERIMENT_ENV,
    'data_sources': {
        'tv_spend': [10, 12, 15, 18, 20],
        'radio_spend': [5, 6, 5, 7, 8],
        'social_spend': [2, 3, 4, 4, 5],
        'marketing_rows': [
            {'tv': 10, 'radio': 5, 'social': 2, 'digital': 3},
            {'tv': 12, 'radio': 6, 'social': 3, 'digital': 4},
            {'tv': 15, 'radio': 5, 'social': 4, 'digital': 4.2},
            {'tv': 18, 'radio': 7, 'social': 4, 'digital': 5},
            {'tv': 20, 'radio': 8, 'social': 5, 'digital': 5.5},
        ],
        'sales': [310, 320, 340, 360, 380],
        'digital_spend': [3, 4, 4.2, 5, 5.5]
    },
    'evaluation': {'metric_ids': ['r2_v1', 'mse_v1'], 'y_true_key': 'y'}
}

submit_resp = requests.post(f"{BASE_URL.rstrip('/')}/runs", json=run_payload, timeout=30)
submit_resp.raise_for_status()
run_ticket = submit_resp.json()
print('Submitted run', run_ticket)

status_url = f"{BASE_URL.rstrip('/')}/runs/{run_ticket['run_id']}"
while True:
    status_resp = requests.get(status_url, timeout=30)
    status_resp.raise_for_status()
    payload = status_resp.json()
    if payload['status'] in {'succeeded', 'failed', 'cancelled'}:
        break
    time.sleep(0.5)

payload


## Clean Up

Remember to delete the temporary CSV if you no longer need it.